In [39]:
import torch
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import BartForConditionalGeneration, BartTokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [3]:
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:500]")

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [5]:
train_test = dataset.train_test_split(test_size=0.1)

In [7]:
train_ds = train_test['train']
test_ds = train_test['test']

In [8]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [9]:
max_input_length = 1024
max_target_length = 128

In [10]:
train_ds

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 450
})

In [11]:
def preprocess_function(examples):
  model_inputs = tokenizer(
      examples['article'],
      max_length=max_target_length,
      truncation=True,
      padding="max_length"
  )

  labels = tokenizer(
      examples['highlights'],
      max_length=max_target_length,
      truncation=True,
      padding="max_length"
  )

  model_inputs['labels'] = labels['input_ids']
  return model_inputs

In [12]:
tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=['article', 'highlights', 'id'])
tokenized_test = test_ds.map(preprocess_function, batched=True, remove_columns=['article', 'highlights', 'id'])

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [30]:
rouge = evaluate.load("rouge")

In [46]:
def compute_metrics(eval_pred):
  preds, labels = eval_pred

  decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

  labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
  decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

  results = rouge.compute(predictions=decoded_preds, references=decoded_labels)

  return {k: round(v, 4) for k, v in results.items()}

In [47]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_summarizer",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=True if torch.cuda.is_available() else False,
    predict_with_generate=True,
    report_to="none"
)

In [48]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

In [49]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,2.427641,1.967450,0.219300,0.078500,0.174900,0.200500
2,1.275219,1.220052,0.209300,0.073800,0.158300,0.190500
3,1.172162,1.168149,0.234600,0.078600,0.175400,0.206900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=339, training_loss=1.9457868024662883, metrics={'train_runtime': 132.6412, 'train_samples_per_second': 10.178, 'train_steps_per_second': 2.556, 'total_flos': 102893027328000.0, 'train_loss': 1.9457868024662883, 'epoch': 3.0})

In [53]:
def summarize(article):
  inputs = tokenizer(
      article,
      return_tensors="pt",
      max_length=max_target_length,
      truncation=True,
  ).to(model.device)

  summarize_ids = model.generate(
      inputs['input_ids'],
      max_length=max_target_length,
      num_beams=4,
      early_stopping=True,
      length_penalty=2.0
  )

  summary = tokenizer.decode(summarize_ids[0], skip_special_tokens=True)
  return summary

In [54]:
sample_article = """
NASA's Perseverance rover has made a groundbreaking discovery on Mars, finding organic molecules in a rock that could potentially point to ancient microbial life.

The rover, which landed in Jezero Crater in February 2021, drilled into a rock nicknamed "Cheyava Falls" in July 2024. Scientists announced on Thursday that the rock contains organic compounds, including carbon-based molecules that are often associated with life on Earth.

"What we see here is a rock that has all the characteristics we look for when searching for signs of ancient life," said Dr. Ken Farley, the project scientist for Perseverance. "It has organic molecules, it was formed in a watery environment, and it shows evidence of chemical reactions that could have provided energy for microbes."

The rock also contains calcium sulfate veins and reddish spots that may have been created by chemical reactions involving iron. These features suggest that water once flowed through the area, creating conditions that could have supported microbial life billions of years ago.

However, scientists caution that organic molecules can also be created through non-biological processes. Further analysis will be needed, including samples that Perseverance is collecting for a future mission to bring back to Earth.

This discovery comes as NASA prepares for the Mars Sample Return mission, which aims to bring Martian rocks back to Earth for detailed laboratory study. The mission is considered critical for determining whether life ever existed on the Red Planet.

Perseverance has already collected 24 rock samples since landing, with plans to gather dozens more before the end of its mission.
"""

print(summarize(sample_article))

NASA's Perseverance rover finds organic molecules in rock that could potentially point to ancient microbial life .
Scientists say the rock contains carbon-based molecules that are often associated with life on Earth .
The rover will land in Jezero Crater in February 2021 .
